# Exporting Data from OSM


[OpenStreetMap](https://www.openstreetmap.org/) is a large, open geospatial database built collaboratively by its users. It is sometimes called the "Wikipedia of maps", since anyone can contribute features.

**OpenStreetMap (OSM)** data is released under the **Open Database License (ODbL 1.0)**. This means it can be freely downloaded, modified, extended, and even used in commercial projects, provided that the source is credited:

© OpenStreetMap contributors

All features in OSM follow a strict structure and are described using tags — key–value pairs that define the characteristics and purpose of each feature.

> In this section, we will explore the OSM data structure, understand how tags are used to describe features, and learn how to download specific features within a defined area.


## 0. Importing Libraries


In [ ]:
import osmnx as ox

# keep all downloaded OSM responses in one cache folder at the root of the repository
ox.settings.cache_folder = "../../cache"


- [**OSMnx**](https://osmnx.readthedocs.io/) (`osmnx`) — a Python library for downloading, analysing, and visualising **OpenStreetMap** data.
  It makes it straightforward to retrieve spatial features by place name or by coordinates.

OSMnx caches every server response, so repeating a query returns the result instantly instead of downloading it again. The line above points that cache at a single `cache/` folder in the root of the repository — without it, every notebook would create a cache folder of its own.

## 1. Geocoding a Boundary

Before downloading data from **OpenStreetMap**, we need to define the study area.
One of the simplest approaches is to specify a place name.

The **OSMnx** library provides the `geocode_to_gdf()` function for this purpose: it geocodes the place name and returns its boundary as a `GeoDataFrame`.

In [ ]:
area_name = "Tsentralny District, Saint Petersburg"

area_osm = ox.geocode_to_gdf(area_name)

Let's visualise the data on a map using the `explore()` method:


In [ ]:
area_osm.explore(tiles="cartodbpositron")

We have successfully retrieved the correct administrative boundary for the district. This means we can now use the same place name to download specific features or the street network from OpenStreetMap.


## 2. Retrieving Data from OpenStreetMap

### 2.1. Tags

In **OpenStreetMap**, all features are described using **tags** — key–value pairs that define the type and characteristics of each feature.

Tags are what allow us to filter the data: we specify which features we are interested in and download them for the previously defined area.

A full list of OSM tags is available in the documentation:
[Map Features](https://wiki.openstreetmap.org/wiki/Map_features)

**How to specify tags in OSMnx**

In OSMnx, tags are passed as a Python dictionary, where:

- the dictionary key is the OSM tag key;
- the value is either a specific tag value, or `True` if we want all features with that key.

Examples:

All features tagged as buildings:

```python
tags = {"building": True}
```

Cafés only:

```python
tags = {"amenity": "cafe"}
```

Cafés, restaurants, and bars:

```python
tags = {"amenity": ["cafe", "restaurant", "bar"]}
```

This allows you to control exactly which features are retrieved.

Let's store the tags for the features we want to work with in the `tags` variable.


In [ ]:
tags = {"building": True}

Now that we understand how tags are defined and how to build an OSM query, let's look at the different ways to download features from OpenStreetMap.


### 2.2. Downloading Data by Area


#### 2.2.1. By Place Name

The simplest way to download data is to query by place name.
If an area's boundary is properly mapped in **OpenStreetMap** — as we verified for our district above — that name can be used directly to retrieve the features of interest.

This approach is particularly convenient when working with administrative units such as districts, cities, or regions, and you need to quickly retrieve all features within their boundaries.


In [ ]:
osm_data = ox.features_from_place(area_name, tags)

Let's display the first five rows of the attribute table using the `head()` method:


In [ ]:
osm_data.head()

Let's visualise the data on a map using the `explore()` method.

The district contains several thousand buildings, so we map only the first 500 features to keep the notebook light — remove `.head(500)` to display them all:

In [ ]:
osm_data.head(500).explore(tiles="cartodbpositron", tooltip=None)

#### 2.2.2. By Polygon Boundary

Another way to download data is to use a **pre-existing geometry**.
Instead of a place name, the query receives a specific polygon that defines the boundary of the area of interest.

This approach is especially useful when the boundary has been drawn manually, refined by the user, or does not correspond to any official administrative division.

To avoid downloading new data, we will reuse the district boundary retrieved in the first section. The result will be identical to the place-name query, since the same area is used — just specified as a polygon.

For that reason the code below is left commented out: uncomment it to run the query yourself.

In [ ]:
# district_polygon = area_osm.geometry.iloc[0]  # district polygon
# osm_data_polygon = ox.features_from_polygon(district_polygon, tags)

# osm_data_polygon.explore(tiles="cartodbpositron", tooltip=None)

#### 2.2.3. By Bounding Box

Another way to define an area is by using a **bounding box**.

A bounding box is the smallest rectangle that fully encloses the selected geometry. In OSMnx it is defined by four coordinates in the order `(west, south, east, north)` — exactly the order returned by the `total_bounds` attribute (`[minx, miny, maxx, maxy]`) that we used in the previous notebook, *Exploring a Dataset*.

Let's derive the bounding box from the district boundary retrieved in the first section, then download the data within it and compare the result with the place-name query.

In [ ]:
osm_data_bbox = ox.features_from_bbox(area_osm.total_bounds, tags)

# compare the number of features returned by the two queries
print(f"By place name: {len(osm_data)}")
print(f"By bounding box: {len(osm_data_bbox)}")

As the counts show, the bounding box query returns noticeably more features. The rectangle also covers territory outside the district boundary, and every building inside that rectangle is returned. If you need features strictly within the boundary, query by place name or by polygon, or clip the result afterwards.

## 3. Exploring the Data

After downloading features from OpenStreetMap, it is important to carry out an initial review of the data.
This helps assess its structure, volume, and quality before proceeding with further analysis.


### 3.1. Number of Features

Let's start by checking how many features were downloaded:


In [ ]:
len(osm_data)

### 3.2. Geometry Types

Let's check which geometry types are present in the data and how many features of each type there are:

In [ ]:
osm_data.geom_type.value_counts()

Building data may contain features with mixed geometry types, because features in OSM are not always mapped using a single geometry type. It is therefore important to identify which geometry types are present before proceeding, and to filter the data if necessary.


### 3.3. Attributes

Let's inspect which attributes the table contains and what data types they have:


In [ ]:
osm_data.dtypes

You may notice that the table has a large number of columns. This is because **OpenStreetMap** stores many tags, and when exporting we receive virtually all attributes that users have added when describing features.


To get a clearer view of the data types across all attributes, let's build a separate table with column names and their corresponding data types:


In [ ]:
dtypes_df = osm_data.dtypes.reset_index()
dtypes_df.columns = ["column", "dtype"]

dtypes_df.head()

## 4. Processing


### 4.1. Filtering by Geometry Type

As we saw earlier, building data can contain mixed geometry types. For most spatial operations, it is important to work with a single geometry type.

Let's keep only features with `Polygon` and `MultiPolygon` geometry:


In [ ]:
osm_data = osm_data[
    osm_data.geom_type.isin(["Polygon", "MultiPolygon"])
]

Let's verify which geometry types remain in the dataset:


In [ ]:
osm_data.geom_type.value_counts()

### 4.2. Validating Geometry

Before proceeding with further analysis, it is important to ensure that all geometries are valid and free from topological errors that could affect the results of spatial operations.


In [ ]:
(~osm_data.geometry.is_valid).sum()

If invalid features are present, there are two options: drop them or repair them.

The first option keeps only the valid features:

In [ ]:
# osm_data = osm_data[osm_data.geometry.is_valid]

The second option repairs the invalid geometry, so that no features are lost. This is the approach we will use here:

In [ ]:
osm_data = osm_data.set_geometry(osm_data.geometry.make_valid())

### 4.3. Preserving Feature Identifiers

In data downloaded from **OpenStreetMap**, the feature identifier (`id`) is not stored as a regular column — it is held in the second level of the `GeoDataFrame` index. It therefore cannot be accessed directly as a table attribute.

For further analysis (filtering, joining tables, saving data), let's extract the feature identifier into a dedicated column.


In [ ]:
osm_data["osm_id"] = osm_data.index.get_level_values("id")

### 4.4. Selecting the Required Attributes

As we noted earlier, the dataset contains a large number of attributes, many of which are not needed for further work. The set of attributes to keep depends on how you intend to use the data.

We will keep only three columns: `osm_id`, `building`, and `geometry`.


In [ ]:
osm_data = osm_data[["osm_id", "building", "geometry"]]

Let's display the first five rows of the attribute table using the `head()` method:


In [ ]:
osm_data.head()

## 5. Saving the Data


We can now save the resulting spatial dataset in any supported format using the `to_file()` method from the GeoPandas library. The path below is relative to the notebook, so the file is written next to it — the shared `data/` folder at the root of the repository holds only the prepared course datasets.

In [ ]:
# osm_data.to_file("osm_build.geojson")

## Summary


In this section, we explored the structure of OpenStreetMap data and the different methods for downloading it using OSMnx.

We defined the study area, built a tag-based query, downloaded spatial features for the target area, and performed an initial review and preprocessing of the data.
